In [142]:
pip install spuco

In [143]:
# Load the SpuCoMNIST dataset
from spuco.datasets import SpuCoMNIST, SpuriousFeatureDifficulty

difficulty = SpuriousFeatureDifficulty.MAGNITUDE_LARGE
classes = [[0,1], [2,3], [4,5], [6,7], [8,9]]

training = SpuCoMNIST(root='data', spurious_feature_difficulty=difficulty, split='train', spurious_correlation_strength=0.99, classes=classes)
test = SpuCoMNIST(root='data', spurious_feature_difficulty=difficulty, split='test', classes=classes)

training.initialize()
test.initialize()

In [144]:
# Create LeNet for ERM and initialize ERM instance
from spuco.models import SupportedModels, model_factory
from spuco.robust_train import ERM
import torch


model_arch = SupportedModels.LeNet
model = model_factory(arch=model_arch, input_shape=training[0][0].shape, num_classes=len(classes))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
erm_instance = ERM(model=model, trainset=training, optimizer=optimizer, batch_size=64, num_epochs=10, device=device)
print(type(erm_instance))


<class 'spuco.robust_train.erm.ERM'>


In [145]:
# Train the model using ERM
erm_instance.train()


In [146]:
# Evaluate the model on the test set
from spuco.evaluate import Evaluator

evaluator = Evaluator(testset=test, group_partition=test.group_partition, group_weights=test.group_weights, batch_size=64, model=model, device=device)
evaluator.evaluate()

Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:19<00:00,  1.25it/s]


{(0, 0): 100.0,
 (0, 1): 80.61465721040189,
 (0, 2): 78.01418439716312,
 (0, 3): 60.283687943262414,
 (0, 4): 54.60992907801418,
 (1, 0): 79.95110024449878,
 (1, 1): 100.0,
 (1, 2): 82.3529411764706,
 (1, 3): 64.70588235294117,
 (1, 4): 65.93137254901961,
 (2, 0): 68.26666666666667,
 (2, 1): 62.93333333333333,
 (2, 2): 100.0,
 (2, 3): 61.6,
 (2, 4): 10.16042780748663,
 (3, 0): 91.4572864321608,
 (3, 1): 68.51385390428212,
 (3, 2): 59.69773299748111,
 (3, 3): 99.74811083123426,
 (3, 4): 74.55919395465995,
 (4, 0): 72.544080604534,
 (4, 1): 25.944584382871536,
 (4, 2): 28.71536523929471,
 (4, 3): 32.323232323232325,
 (4, 4): 100.0}

In [147]:
# Overall accuracy of the model is high for majority groups. However, it clearly is predicting 
# based on the spurious attributes rather than the predictive features, as evidenced by the low 
# accuracy on the minority groups.  

print("Spurious correlation accuracy:", evaluator.evaluate_spurious_attribute_prediction())
print("Average accuracy:", evaluator.average_accuracy)
print("Worst group accuracy:", evaluator.worst_group_accuracy)

Spurious correlation accuracy: 19.65
Average accuracy: 69.14999999999999
Worst group accuracy: ((2, 4), 10.16042780748663)


In [ ]:
# Spurious correlation accuracy isn't too high, but we can still see that the model 
# struggles with predicting samples whose spurious features are absent in test 
# data.Let's implement step 2 of the pipeline and see if we can improve model performance 
# on minority groups. 

# Cluster input data based on penultimate layer features using KMeans. 

from spuco.group_inference import Cluster
from spuco.utils import get_model_outputs

# Use an underfit LeNet to extract features
uf_model = model_factory(arch=model_arch, input_shape=training[0][0].shape, num_classes=len(classes))
uf_model.to(device)
uf_optimizer = torch.optim.Adam(uf_model.parameters(), lr=0.001)
uf_model_instance = ERM(model=uf_model, trainset=training, batch_size=64, optimizer=uf_optimizer, num_epochs=1, device=device)
uf_model_instance.train()

# Get features from the penultimate layer of the underfit model and cluster them using KMeans.
Z = get_model_outputs(model=uf_model, dataset=training, device=device, features=True)
cluster_instance = Cluster(Z=Z,class_labels=training.labels, num_clusters=5, device=device)
inferred_groups = cluster_instance.infer_groups()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [155]:
# Step 3: Use clusters as predictions for (class_label, spurious_attribute) groups and use
# them to train a model using GroupBalanceBatchERM.
from spuco.robust_train import GroupBalanceBatchERM

gb_model = model_factory( arch=model_arch, input_shape=training[0][0].shape, num_classes=len(classes))
gb_model.to(device)
gb_optimizer = torch.optim.Adam(gb_model.parameters(), lr=0.001)

gb_instance = GroupBalanceBatchERM(model=gb_model, trainset=training, group_partition = inferred_groups, batch_size=64, optimizer=gb_optimizer, num_epochs=10, device=device)
gb_instance.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [156]:
# Evaluate gb model on the test set
gb_evaluator = Evaluator(testset=test, group_partition=test.group_partition, group_weights=test.group_weights, batch_size=64, model=gb_model, device=device)
gb_evaluator.evaluate()


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:20<00:00,  1.22it/s]


{(0, 0): 99.52718676122932,
 (0, 1): 91.25295508274232,
 (0, 2): 91.725768321513,
 (0, 3): 96.21749408983452,
 (0, 4): 93.3806146572104,
 (1, 0): 86.0635696821516,
 (1, 1): 99.26650366748166,
 (1, 2): 91.17647058823529,
 (1, 3): 86.76470588235294,
 (1, 4): 92.6470588235294,
 (2, 0): 96.26666666666667,
 (2, 1): 90.93333333333334,
 (2, 2): 99.73333333333333,
 (2, 3): 99.2,
 (2, 4): 77.27272727272727,
 (3, 0): 88.94472361809045,
 (3, 1): 77.32997481108312,
 (3, 2): 89.4206549118388,
 (3, 3): 98.74055415617129,
 (3, 4): 88.91687657430731,
 (4, 0): 90.68010075566751,
 (4, 1): 67.00251889168766,
 (4, 2): 65.99496221662469,
 (4, 3): 88.38383838383838,
 (4, 4): 98.98989898989899}

In [ ]:
# Model accuracy has improved drastically on the minority groups. Classifier is no longer 
# as biased towards the spurious features and is able to learn predictive features as well.

print("Spurious correlation accuracy:", gb_evaluator.evaluate_spurious_attribute_prediction())
print("Average accuracy:", gb_evaluator.average_accuracy)
print("Worst group accuracy:", gb_evaluator.worst_group_accuracy)

Spurious correlation accuracy: 20.37
Average accuracy: 89.86999999999999
Worst group accuracy: ((4, 2), 65.99496221662469)
